[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C65_ProblemSolving_Communication_Course/02_diagnosis/02_diagnosis_reasoning.ipynb)

# 02 · 诊断与归因推理（搜索策略 / 信息增益证明 / 可证伪假设 / 排除法 / 混杂因素 / 三个诊断案例）

目标：把「猜哪里错了」从直觉，变成一套**可以运行、可以断言的诊断方法论**。

本 notebook 你会亲手实现：
1. **诊断的通用框架** —— 症状 → 假设集 → 检查(成本+划分) → 按策略排序 → 执行 → 收敛
2. **信息增益的数学证明** —— 严格推导并数值验证「最优检查是把候选概率质量切成 50/50 那个」
3. **可证伪性检验器** —— 把「可能是 X」和「如果是 X，那么 Y 应该成立」用代码区分开
4. **排除法与反例构造** —— 一次检查同时排除多个假设，以及主动构造反例检验断言
5. **混杂因素模拟** —— 复现经典的辛普森悖论，展示「相关不等于因果」
6. **三个诊断案例分步演练** —— 线上指标掉了 / 训练不收敛 / 离线好线上差

> 心智模型：**诊断题考的不是「猜得准不准」，是「你的搜索过程能不能被复现、被审查」。**

## 1 · 诊断的通用框架：三种策略的代码化对比

「最便宜优先」和「信息增益最大」的核心区别，在假设先验不均匀时会给出完全不同的排序。
下面用一个具体的四选一诊断场景（线上指标下降的四个嫌疑假设）把这个分歧显式地算出来。

In [ ]:
import math
from collections import defaultdict

def binary_entropy(w):
    """二元检查把当前假设的概率质量切成 w : (1-w) 两组时能拿到的信息增益（bit）。
    这是熵的分组公理给出的恒等式：无论组内构成如何，期望信息增益只取决于这个切分比例 w。"""
    if w <= 0.0 or w >= 1.0:
        return 0.0
    return -(w * math.log2(w) + (1 - w) * math.log2(1 - w))

# 四个候选检查：D 便宜但几乎不排除任何假设；A/C/B 各自的切分比例 w 与成本不同
CHECKS = {
    'D_探路式检查（几乎不排除任何假设）': {'cost': 0.3, 'w': 0.01},
    'A_数据管线日志（能砍掉一半概率质量）': {'cost': 0.4, 'w': 0.50},
    'C_评测脚本diff（只覆盖小先验的假设）': {'cost': 0.5, 'w': 0.20},
    'B_模型单元测试（最贵，切分也不均衡）': {'cost': 3.0, 'w': 0.30},
}
for c in CHECKS.values():
    c['gain'] = binary_entropy(c['w'])
    c['gain_per_cost'] = c['gain'] / c['cost']

cheapest_first = sorted(CHECKS, key=lambda k: CHECKS[k]['cost'])
info_gain_max  = sorted(CHECKS, key=lambda k: -CHECKS[k]['gain_per_cost'])

print(f"{'检查':<40}{'cost':>6}{'w':>6}{'gain':>8}{'gain/cost':>11}")
for name in CHECKS:
    c = CHECKS[name]
    print(f"{name:<40}{c['cost']:>6.2f}{c['w']:>6.2f}{c['gain']:>8.4f}{c['gain_per_cost']:>11.4f}")

print('\n最便宜优先的顺序   :', cheapest_first)
print('信息增益最大的顺序 :', info_gain_max)

assert cheapest_first == ['D_探路式检查（几乎不排除任何假设）', 'A_数据管线日志（能砍掉一半概率质量）',
                          'C_评测脚本diff（只覆盖小先验的假设）', 'B_模型单元测试（最贵，切分也不均衡）']
assert info_gain_max == ['A_数据管线日志（能砍掉一半概率质量）', 'C_评测脚本diff（只覆盖小先验的假设）',
                         'B_模型单元测试（最贵，切分也不均衡）', 'D_探路式检查（几乎不排除任何假设）']
print('\n✅ 两种策略给出完全相反的排序：D 检查最便宜，「最便宜优先」会让你第一步就查它，')
print('   但它的信息量小到几乎没用——「信息增益最大」策略正确地把它排到了最后一位。')

## 2 · 信息增益的数学证明：50/50 切分为什么是最优检查

先在 0.01–0.99 间扫描上百个可能的切分比例，数值验证 `binary_entropy(w)` 在 w=0.5 处取最大值、恰好等于 1 bit；
再从「候选个数」的等价视角（n 个等概率假设切成 k:(n-k)）交叉核对，证明两种描述本质是同一件事。

In [ ]:
# ── 视角一：概率质量切分比例 w ──
ws = [i / 100 for i in range(1, 100)]
gains = [binary_entropy(w) for w in ws]
best_w = ws[gains.index(max(gains))]

assert abs(best_w - 0.5) < 1e-9
assert abs(max(gains) - 1.0) < 1e-9
print(f'扫描 99 个候选切分点 w ∈ [0.01, 0.99]：信息增益最大的切分点是 w={best_w}，增益恰好 = {max(gains):.6f} bit。')

# ── 视角二：n 个等概率候选，切成 k : (n-k) 两组 ──
def gain_for_count_split(n, k):
    """n 个等概率假设，切成大小 k 与 n-k 两组的期望信息增益。"""
    prior = math.log2(n)
    post = 0.0
    if k > 0:
        post += (k / n) * math.log2(k)
    if n - k > 0:
        post += ((n - k) / n) * math.log2(n - k)
    return prior - post

def best_k(n):
    return max(range(1, n), key=lambda k: gain_for_count_split(n, k))

for n in (5, 6, 7, 8, 9, 13):
    k = best_k(n)
    assert k in (n // 2, (n + 1) // 2), (n, k)
    # 两种视角必须完全对上：count-split 的增益 == 用 w=k/n 代入 binary_entropy 的增益
    assert abs(gain_for_count_split(n, k) - binary_entropy(k / n)) < 1e-9
    print(f'n={n:>2} 个候选，最优切分点 k={k}（最接近 n/2），增益={gain_for_count_split(n, k):.4f} bit')

print('\n✅ 无论用「概率质量比例」还是「候选个数」描述，「最优检查是把候选切成 50/50 那个」都成立，')
print('   而且恰好换来 1 bit 信息——这正是 C61-03「训练链路八段二分定位」每一步都在中点检查的数学原因。')

## 3 · 可证伪性：区分「听起来像分析」和「真能被检验」

一个不可证伪的假设，不管给它什么证据都会"成立"——它没有信息量。
下面用一对对照（模糊版 vs 可证伪版）把这个区别显式地跑一遍。

In [ ]:
# 模糊版："可能是数据问题"——不管看到什么证据都"成立"
def vague_hypothesis_predicts(evidence):
    return True

# 可证伪版："如果是数据问题，那么这个类别在训练集里的实例数应该明显偏少"
TRAIN_COUNT_THRESHOLD = 50
def falsifiable_hypothesis_predicts(evidence):
    return evidence['count_in_train'] < TRAIN_COUNT_THRESHOLD

possible_evidence = [{'count_in_train': n} for n in (5, 20, 49, 50, 80, 500)]

vague_results = [vague_hypothesis_predicts(e) for e in possible_evidence]
good_results  = [falsifiable_hypothesis_predicts(e) for e in possible_evidence]

print('可能证据下的预测（模糊版）  :', vague_results)
print('可能证据下的预测（可证伪版）:', good_results)

assert all(vague_results), '不可证伪的假设，无论证据是什么，"预测"永远成立——这正是它没有信息量的证据'
assert not all(good_results), '可证伪的假设，存在某些可能证据会让它的预测落空'
print('\n✅ 区分标准：把"可能是 X"改写成"如果是 X，那么 Y 应该成立"——')
print('   如果找不到任何一种可能证据会让 Y 不成立，这句话就没有诊断价值。')

## 4 · 排除法与反例构造：一次检查同时裁决多个假设

三个假设各自的"如果为真，证据应该长什么样"互不冲突，一次拉取证据就能同时判三个，
而不是排队一个一个单独去查。

In [ ]:
HYPOTHESES = {
    '数据标签错位（坐标系反了）': lambda ev: ev['bbox_valid_ratio'] < 0.5,
    '模型欠拟合':                 lambda ev: ev['train_loss'] > 2.0,
    '评测脚本口径错误':           lambda ev: ev['offline_metric'] != ev['manual_recompute'],
}
EVIDENCE = {'bbox_valid_ratio': 0.97, 'train_loss': 0.35, 'offline_metric': 0.812, 'manual_recompute': 0.734}

def eliminate_by_evidence(hypotheses, evidence):
    """保留"预测与证据相符（未被证据反驳）"的假设，其余判定为已排除。"""
    survivors = {name: pred for name, pred in hypotheses.items() if pred(evidence)}
    eliminated = [name for name in hypotheses if name not in survivors]
    return survivors, eliminated

survivors, eliminated = eliminate_by_evidence(HYPOTHESES, EVIDENCE)
print('剩下的假设：', list(survivors))
print('被排除的假设：', eliminated)

assert eliminated == ['数据标签错位（坐标系反了）', '模型欠拟合']
assert list(survivors) == ['评测脚本口径错误']
print('\n✅ 三个假设查一次证据就排除了两个——排除法的价值在于每一次检查都是"淘汰赛"，不是"确认赛"。')

In [ ]:
# 反例构造：面试官抛出一个"看起来对"的断言，主动想一个可能推翻它的反例来检验
claim_holds = lambda size_cm, is_false_positive: not (size_cm > 60 and is_false_positive)  # 断言：超过 60cm 不会误检

false_positive_log = [(45, True), (30, True), (68, True), (55, False)]   # (标志尺寸cm, 是否误检)
counterexamples = [(s, fp) for s, fp in false_positive_log if not claim_holds(s, fp)]

print(f'找到反例：{counterexamples}')
assert counterexamples == [(68, True)]
print('✅ 一个 68cm 的标志仍然被误检，断言"超过 60cm 不会误检"不成立——')
print('   这正是"先想反例、再相信断言"的价值：不用等对方给反例，自己主动构造。')

## 5 · 混杂因素：复现经典的辛普森悖论

用肾结石治疗方案的经典数字（Simpson's paradox 教科书例子），套到"新模型 A vs 旧模型 B 在两类场景下的表现"，
展示"每个子场景 A 都赢，合计却是 B 赢"这个反直觉现象。

In [ ]:
DATA = {
    '小场景（近距离/大目标）': {'A': (81, 87), 'B': (234, 270)},
    '大场景（远距离/小目标）': {'A': (192, 263), 'B': (55, 80)},
}

def rate(success_total):
    s, t = success_total
    return s / t

for scene, d in DATA.items():
    ra, rb = rate(d['A']), rate(d['B'])
    print(f"{scene:<18} A={ra:.3f}  B={rb:.3f}  {'A 更好' if ra > rb else 'B 更好'}")

agg_a = tuple(sum(x) for x in zip(*[d['A'] for d in DATA.values()]))
agg_b = tuple(sum(x) for x in zip(*[d['B'] for d in DATA.values()]))
ra_all, rb_all = rate(agg_a), rate(agg_b)
print(f"{'合计':<18} A={ra_all:.3f}  B={rb_all:.3f}  {'A 更好' if ra_all > rb_all else 'B 更好'}")

assert rate(DATA['小场景（近距离/大目标）']['A']) > rate(DATA['小场景（近距离/大目标）']['B'])
assert rate(DATA['大场景（远距离/小目标）']['A']) > rate(DATA['大场景（远距离/小目标）']['B'])
assert ra_all < rb_all
print('\n✅ 两个子场景里 A 都赢，合计却是 B 赢——这就是辛普森悖论：')
print('   不看混杂因素（这里是"场景难度分布"）就直接比较合计指标，会得出反向结论。')

## 6 · 三个诊断案例分步演练：线上指标掉了 / 训练不收敛 / 离线好线上差

复用第 1 节的 `binary_entropy` 与排序思路，封装成 `rank_by_gain_per_cost`，
在三个真实场景里各自跑一遍"信息增益最大"策略，并核对推荐的首选检查是否符合直觉。

In [ ]:
def rank_by_gain_per_cost(checks):
    """checks: {检查名: {'cost':成本, 'w':能把当前假设概率质量切成的比例}}
       返回按 gain/cost 降序排列的检查名列表。"""
    scored = {name: binary_entropy(c['w']) / c['cost'] for name, c in checks.items()}
    return sorted(scored, key=lambda k: -scored[k])

# ── 案例①：线上指标掉了 ──
case1_checks = {
    '检查线上模型版本号与预期是否一致': {'cost': 0.2, 'w': 0.25},
    '查基础设施监控面板（超时率/丢帧率）': {'cost': 0.3, 'w': 0.15},
    '对比评测脚本近两周 diff': {'cost': 0.5, 'w': 0.20},
    '对比本周与上周的输入特征分布（PSI/KS检验）': {'cost': 1.0, 'w': 0.40},
}
order1 = rank_by_gain_per_cost(case1_checks)
print('案例① 线上指标掉了 —— 推荐检查顺序：')
for name in order1:
    c = case1_checks[name]
    print(f"  {name:<40} cost={c['cost']:<4} gain/cost={binary_entropy(c['w'])/c['cost']:.3f}")
assert order1[0] == '检查线上模型版本号与预期是否一致'
assert order1[-1] == '对比本周与上周的输入特征分布（PSI/KS检验）'
print('  → 版本检查几乎零成本又能排除一个高优先级假设，理应第一个做；分布对比虽然信息量不小，但太贵，排最后。\n')

In [ ]:
# ── 案例②：训练不收敛（呼应 C61-03，本课不重复具体病例，只做排序演示） ──
case2_checks = {
    '检查 DataLoader 是否 shuffle=True 且抽样验证 batch 内类别多样性': {'cost': 0.2, 'w': 0.25},
    '单 batch 过拟合测试（应能逼近 loss=0，见 C61-03）': {'cost': 0.5, 'w': 0.25},
    '打印各层激活/梯度的方差看是否量级异常': {'cost': 0.6, 'w': 0.20},
    '跑 lr range test 看 loss vs lr 曲线': {'cost': 0.8, 'w': 0.30},
}
order2 = rank_by_gain_per_cost(case2_checks)
print('案例② 训练不收敛 —— 推荐检查顺序：')
for name in order2:
    c = case2_checks[name]
    print(f"  {name:<50} cost={c['cost']:<4} gain/cost={binary_entropy(c['w'])/c['cost']:.3f}")
assert order2[0] == '检查 DataLoader 是否 shuffle=True 且抽样验证 batch 内类别多样性'
print('  → 查 shuffle 几乎零成本，理应先于代价更高的 lr range test；这正是"最便宜优先"在假设先验接近时的直觉，')
print('    但这里用的是信息增益/成本的通用公式算出来的，不是死记"先查 shuffle"这条规则。\n')

In [ ]:
# ── 案例③：离线好线上差（呼应 C60 的预处理一致性、C63-02 的泄漏检测） ──
case3_checks = {
    '对拍：同一张图分别过训练预处理与车端预处理，比较像素差（见 C60-01）': {'cost': 0.4, 'w': 0.35},
    '重新跑一次分组划分的四类泄漏检测（见 C63-02）': {'cost': 0.3, 'w': 0.15},
    '检查离线测试集的采集时间/设备/地点是否覆盖线上流量画像': {'cost': 0.6, 'w': 0.20},
    '抽取线上 badcase 人工 review，看是否是训练分布之外的场景': {'cost': 1.0, 'w': 0.30},
}
order3 = rank_by_gain_per_cost(case3_checks)
print('案例③ 离线好线上差 —— 推荐检查顺序：')
for name in order3:
    c = case3_checks[name]
    print(f"  {name:<55} cost={c['cost']:<4} gain/cost={binary_entropy(c['w'])/c['cost']:.3f}")
assert order3[0] == '对拍：同一张图分别过训练预处理与车端预处理，比较像素差（见 C60-01）'
print('  → 预处理对拍便宜且信息量大，理应最先做；人工 review badcase 虽然常被第一反应想到，')
print('    但成本高、且不如直接对拍来得精确，被正确排到了后面。')

## ✏️ 练习 1：检查排序器（带数值输出）

实现 `rank_checks_with_scores(checks)`：返回 `[(名字, gain, gain_per_cost), ...]`，
按 `gain_per_cost` 降序排列——不是只返回名字，要带上数值，方便面试时口头报出"这个检查的性价比是多少"。

In [ ]:
def rank_checks_with_scores(checks):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
toy = {
    'X': {'cost': 1.0, 'w': 0.5},
    'Y': {'cost': 2.0, 'w': 0.5},
    'Z': {'cost': 0.5, 'w': 0.1},
}
ranked = rank_checks_with_scores(toy)
names = [r[0] for r in ranked]
assert names == ['X', 'Z', 'Y'], names   # X: 1.0/1=1.0 | Z: H(0.1)/0.5≈0.938 | Y: 1.0/2=0.5
for name, gain, gpc in ranked:
    assert abs(gpc - gain / toy[name]['cost']) < 1e-9
assert abs(ranked[0][1] - 1.0) < 1e-9    # X 的 gain 恰好是 1 bit（50/50 切分）
print(ranked)
print('✅ 练习 1 通过：性价比排序不是只看谁便宜（Y 比 Z 贵却因为切分更均衡而排在 Z 后面才对吗？——')
print('   注意这里是 Z 排在 Y 前面，因为 Z 虽然切分不如 X 均衡，但成本低到弥补了这个劣势。')

## ✏️ 练习 2：二分定位器（呼应 C61-03 的训练链路排查）

实现 `bisect_find_fault(n_stages, is_broken)`：`is_broken(i)` 对下标 `i` 返回 True/False，
且满足**单调性**假设——一旦某个阶段坏了，它和它之后的所有阶段都会返回 True。
返回 `(first_broken_index_or_None, num_calls)`，且 `num_calls` 不应超过 `ceil(log2(n_stages)) + 1`。

In [ ]:
def bisect_find_fault(n_stages, is_broken):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def make_oracle(fault_at):
    calls = [0]
    def is_broken(i):
        calls[0] += 1
        return i >= fault_at
    return is_broken, calls

n = 16
for fault_at in (0, 1, 7, 15, 16):   # 16 表示"没坏"
    oracle, calls = make_oracle(fault_at)
    result, num_calls = bisect_find_fault(n, oracle)
    expected = fault_at if fault_at < n else None
    assert result == expected, (fault_at, result, expected)
    assert num_calls <= math.ceil(math.log2(n)) + 1, (fault_at, num_calls)

print(f'n={n} 段流水线，二分定位最多用了 {math.ceil(math.log2(n)) + 1} 次检查（而不是最坏 {n} 次线性扫描）')
print('✅ 练习 2 通过：二分法要求"单调性"（坏了就一直坏）——这正是它和信息增益策略的适用条件差异：')
print('   流水线阶段有天然顺序时用二分法，候选是一堆并列假设时用信息增益/成本排序。')

## ✏️ 练习 3：可证伪性检验器

实现 `is_falsifiable(predict_fn, possible_evidence)`：如果 `possible_evidence` 中存在至少一个元素
让 `predict_fn` 返回 `False`，则可证伪，返回 `True`；如果对所有可能证据都返回 `True`（永真式），
则不可证伪，返回 `False`。

In [ ]:
def is_falsifiable(predict_fn, possible_evidence):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert is_falsifiable(vague_hypothesis_predicts, possible_evidence) is False
assert is_falsifiable(falsifiable_hypothesis_predicts, possible_evidence) is True

always_false = lambda ev: False
assert is_falsifiable(always_false, possible_evidence) is True   # 存在"返回 False"的证据（其实全是），满足定义

empty_possible = []
assert is_falsifiable(falsifiable_hypothesis_predicts, empty_possible) is False  # 没有可能证据可供检验，视为不可证伪
print('✅ 练习 3 通过：可证伪性不是"证明它对"，是"存在会让它错的可能证据"。')

## ✏️ 练习 4：辛普森悖论检测器

实现 `detect_simpson(subgroup_data)`：`subgroup_data` 形如
`{子群体名: {'A': (成功数, 总数), 'B': (成功数, 总数)}}`。返回
`{'subgroup_winners': {子群体名: 'A'/'B'}, 'overall_winner': 'A'/'B', 'is_paradox': bool}`，
其中 `is_paradox` 表示"所有子群体的赢家一致，但整体赢家与之相反"。

In [ ]:
def detect_simpson(subgroup_data):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
result = detect_simpson(DATA)
assert result['subgroup_winners'] == {'小场景（近距离/大目标）': 'A', '大场景（远距离/小目标）': 'A'}
assert result['overall_winner'] == 'B'
assert result['is_paradox'] is True

NORMAL = {'g1': {'A': (80, 100), 'B': (60, 100)}, 'g2': {'A': (40, 100), 'B': (20, 100)}}
result2 = detect_simpson(NORMAL)
assert result2['is_paradox'] is False
assert result2['overall_winner'] == 'A'
print(result)
print(result2)
print('✅ 练习 4 通过：把"分层看谁赢"和"合计看谁赢"分开算，才能自动检出混杂因素造成的反转。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def rank_checks_with_scores(checks):
    scored = []
    for name, c in checks.items():
        gain = binary_entropy(c['w'])
        scored.append((name, gain, gain / c['cost']))
    scored.sort(key=lambda t: -t[2])
    return scored

In [ ]:
# 练习 2 参考答案
def bisect_find_fault(n_stages, is_broken):
    calls = 1
    if not is_broken(n_stages - 1):
        return None, calls
    lo, hi = 0, n_stages - 1
    while lo < hi:
        mid = (lo + hi) // 2
        calls += 1
        if is_broken(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo, calls

In [ ]:
# 练习 3 参考答案
def is_falsifiable(predict_fn, possible_evidence):
    return any(not predict_fn(e) for e in possible_evidence)

In [ ]:
# 练习 4 参考答案
def detect_simpson(subgroup_data):
    def rate(st):
        s, t = st
        return s / t
    subgroup_winners = {}
    agg = {'A': [0, 0], 'B': [0, 0]}
    for name, d in subgroup_data.items():
        ra, rb = rate(d['A']), rate(d['B'])
        subgroup_winners[name] = 'A' if ra > rb else 'B'
        agg['A'][0] += d['A'][0]; agg['A'][1] += d['A'][1]
        agg['B'][0] += d['B'][0]; agg['B'][1] += d['B'][1]
    overall_a, overall_b = rate(tuple(agg['A'])), rate(tuple(agg['B']))
    overall_winner = 'A' if overall_a > overall_b else 'B'
    winners_set = set(subgroup_winners.values())
    is_paradox = (len(winners_set) == 1) and (overall_winner not in winners_set)
    return {'subgroup_winners': subgroup_winners, 'overall_winner': overall_winner, 'is_paradox': is_paradox}

---
## 🧪 真实工程胶囊：诊断白板模板 + 面试口播要点

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 诊断白板模板（面试当场可以照这个骨架展开）
# ══════════════════════════════════════════════════════════════════════
# 1. 复述症状 + 确认边界：全量还是局部？突然还是渐变？和什么时间点重合？
# 2. 列出 2-4 个假设，每个假设都写成"如果是 X，那么 Y 应该成立"的可证伪形式
# 3. 给每个假设估一个粗略先验 + 每个候选检查估一个成本，选 gain/cost 最高的先做
#    （候选有天然顺序且故障单调 -> 用二分法；候选先验接近 -> 最便宜优先够用）
# 4. 一次检查尽量同时对多个假设下判断（排除法），查完立刻剔除被反驳的假设
# 5. 警惕混杂因素：下因果结论前，先问"还有什么也会同时随它一起变化"
# 6. 预告收敛路径："如果是 A，下一步做 X；如果是 B，下一步做 Y"

# ══════════════════════════════════════════════════════════════════════
# B. 面试里最容易被追问的三句话，提前想好怎么答
# ══════════════════════════════════════════════════════════════════════
# Q: "你的第一步会做什么？"
# A: "我需要先确认两件事：症状是全量的还是局部的、是否和某次上线时间重合。
#     这决定了后面假设集合的形状——先要信息不是拖延，是避免在错误的候选集里瞎查。"
#
# Q: "为什么先查这个而不是别的？"
# A: "我会按'期望信息增益除以查证成本'排序：这个检查虽然不是最便宜的，
#     但它能排除的假设概率质量最大，性价比最高。"
#
# Q: "你怎么知道这不只是相关，而是真的因果？"
# A: "我会先做分层对比，控制住可能的混杂变量（比如场景难度、设备型号），
#     如果分层后结论依然一致，才敢下因果判断——辛普森悖论提醒我们合计指标可能会撒谎。"

# ══════════════════════════════════════════════════════════════════════
# C. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 检测调试的具体病例（NaN/loss躺平/mAP恒0）      -> C61 模块 03（本课只讲抽象方法论）
# · 数据泄漏与分组划分的技术细节                    -> C63 模块 02（本课只引用其结论作为混杂因素案例）
# · 辛普森悖论等统计陷阱的完整问答骨架              -> C64 模块 04
# · 权衡与决策（诊断完根因之后怎么选方案）          -> C65 模块 03（下一模块）
'''
print(RECIPE)
for token in ['期望信息增益', '可证伪', '混杂变量', 'C61 模块 03', 'C63 模块 02', 'C64 模块 04']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：诊断白板骨架 / 高频追问的标准答法 / 与其他课程的分工边界')

### 小结

- **三种搜索策略各有前提**：二分法要求候选有序且故障单调；最便宜优先要求假设先验接近均匀；
  信息增益最大最通用——本质上前两者都是它在特定结构下的特例。
- **信息增益的数学结论很干净**：一个二元检查的期望信息增益恰好等于它把概率质量切分比例 `w` 的二元熵 `H_b(w)`，
  在 `w=0.5` 处取唯一最大值、恰好 1 bit，与检查内部细节无关。
- **假设必须可证伪**：把"可能是 X"改写成"如果是 X，那么 Y 应该成立"，找不出任何会让 Y 落空的可能证据，
  这句话就没有诊断价值。
- **排除法的价值在于"一次检查裁决多个假设"**，而反例构造是把可证伪思路反过来主动用在别人的断言上。
- **相关不等于因果**：辛普森悖论证明"分层看结论一致，合计看结论反转"是真实存在的现象，
  下因果判断前先问"还有什么也在同时变化"。
- **C61-03 的具体调试清单，是本模块三种策略在检测训练这个领域的三个具体实例**——
  换一个领域，方法论不变，只是假设集合的内容换了。
- **面试里"先要更多信息"是加分项，不是减分项**——前提是要得快、要得准，问完就要开始给假设，不能停在澄清阶段不动。

下一站：**模块 03 · 权衡与决策** —— 诊断出根因之后，往往面对好几个修复方案，
下一个最容易在面试里丢分的地方，是选了一个方案却说不清放弃了什么。